# Sentence Window Retrieval [Step 3 - The Sharpest Possible Index]

> **MLCourse - Agentic AI - Advanced RAG - Contextual Retrieval**

Sentence window retrieval pushes the small-to-big idea to its limit: **index
individual sentences**, and when one matches, return it together with the *N*
sentences on either side.

```
   ... s6   s7   [ s8  s9  S10  s11  s12 ]   s13 ...
                    ^-- window of +/-2 around the matched sentence S10
```

Compared with a parent document retriever, the difference is that the returned
context is **centred on the match** rather than being a fixed pre-defined block.
If the answer sits at the very end of a paragraph, a parent retriever gives you
the whole paragraph; a sentence window gives you the end of that paragraph *and*
the start of the next one. For prose where ideas run across paragraph
boundaries, that is the better shape.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# "Parent" units: paragraphs. Big enough to answer from, too big to retrieve
# precisely. These are the documents we will later cut into small children.
parents = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("parent paragraphs:", len(parents))
print("mean parent length:", int(sum(len(p) for p in parents) / len(parents)), "chars")

parent paragraphs: 237
mean parent length: 379 chars


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def build_index(texts):
    """Embed a list of texts and return the normalised matrix."""
    return encoder.encode(texts, normalize_embeddings=True,
                          batch_size=64, show_progress_bar=False)


def search(index, texts, query, top_n=5):
    """Return [(position, score)] of the best matches in `index`."""
    q = encoder.encode([query], normalize_embeddings=True)[0]
    sims = index @ q
    order = np.argsort(sims)[::-1][:top_n]
    return [(int(i), float(sims[i])) for i in order]


print("encoder ready:", encoder.get_sentence_embedding_dimension(), "dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

encoder ready: 384 dimensions


### 2. Split into sentences and keep the order

The critical implementation detail: sentences must be stored as **one flat,
ordered list**, because the window is defined by *position*. Losing the original
order breaks the whole technique.

We use a simple regex splitter here. In production, use a real sentence splitter
(spaCy, NLTK, or `SentenceSplitter` in LlamaIndex) - abbreviations like "Mr." and
"e.g." defeat naive regexes.

In [5]:
full_text = "\n\n".join(parents)

SENT_RE = re.compile(r"(?<=[.!?])\s+")
sentences = [s.strip() for s in SENT_RE.split(full_text) if len(s.strip()) > 25]

print("sentences:", len(sentences))
print("mean length:", int(sum(len(s) for s in sentences) / len(sentences)), "chars\n")
for i in range(40, 44):
    print(f"  [{i}] {sentences[i][:100]}...")

sentences: 484
mean length: 184 chars

  [40] There were doors all round the hall, but they were all locked; and when Alice had been all the way d...
  [41] Suddenly she came upon a little three-legged table, all made of solid glass; there was nothing on it...
  [42] either the locks were too large, or the key was too small, but at any rate it would not open any of ...
  [43] However, on the second time round, she came upon a low curtain she had not noticed before, and behin...


In [6]:
sentence_vectors = build_index(sentences)
print("index:", sentence_vectors.shape, "- one vector per sentence")

index: (484, 384) - one vector per sentence


### 3. The window function

Retrieval is now two steps: find the best sentence, then slice around it. Note
the `max`/`min` clamping - windows at the very start or end of the corpus must
not wrap around, or you will splice unrelated text together.

In [7]:
def sentence_window(query, window=3, top_n=3):
    """Match single sentences, return each with `window` neighbours on each side."""
    hits = search(sentence_vectors, sentences, query, top_n=top_n * 3)

    results, used = [], set()
    for pos, score in hits:
        if any(abs(pos - u) <= window for u in used):
            continue                     # skip overlapping windows
        used.add(pos)
        lo = max(0, pos - window)
        hi = min(len(sentences), pos + window + 1)
        results.append({
            "center": pos,
            "score": score,
            "sentence": sentences[pos],
            "window": " ".join(sentences[lo:hi]),
            "span": (lo, hi),
        })
        if len(results) == top_n:
            break
    return results


QUESTION = "How does the Cheshire Cat disappear?"

for r in sentence_window(QUESTION, window=3, top_n=3):
    print(f"sentence {r['center']} (cosine={r['score']:.3f}), window {r['span']}")
    print(f"  MATCHED : {r['sentence'][:120]}...")
    print(f"  WINDOW  : {r['window'][:230]}...")
    print()

sentence 365 (cosine=0.594), window (362, 369)
  MATCHED : The Cat seemed to think that there was enough of it now in sight, and no more of it appeared....
  WINDOW  : Alice began to feel very uneasy: to be sure, she had not as yet had any dispute with the Queen, but she knew that it might happen any minute, “and then,” thought she, “what would become of me? They’re dreadfully fond of beheading ...

sentence 375 (cosine=0.571), window (372, 379)
  MATCHED : (It was this last remark that had made the whole party look so grave and anxious.)

The Cat’s head began fading away the...
  WINDOW  : The moment Alice appeared, she was appealed to by all three to settle the question, and they repeated their arguments to her, though, as they all spoke at once, she found it very hard indeed to make out exactly what they said. The...

sentence 371 (cosine=0.492), window (368, 375)
  MATCHED : When she got back to the Cheshire Cat, she was surprised to find quite a large crowd collected round it: the

The overlap guard matters. Without it, sentences 100, 101 and 102 all matching
the query would produce three windows covering almost identical text - three
context slots spent on one passage.

### 4. Retrieval sharpness versus the alternatives

Single sentences are the sharpest thing you can embed. Here is the top-hit
cosine for three index granularities on the same query.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

para_vectors = build_index(parents)
chunk_texts = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=80).split_text(full_text)
chunk_vectors = build_index(chunk_texts)

print(f"{'index granularity':<26}{'units':>8}{'best cosine':>14}")
print("-" * 48)
for label, vecs, texts in [("sentences", sentence_vectors, sentences),
                           ("600-char chunks", chunk_vectors, chunk_texts),
                           ("paragraphs", para_vectors, parents)]:
    top = search(vecs, texts, QUESTION, top_n=1)[0]
    print(f"{label:<26}{len(texts):>8}{top[1]:>14.3f}")

index granularity            units   best cosine
------------------------------------------------
sentences                      484         0.594
600-char chunks                217         0.615
paragraphs                     237         0.615


### 5. Tuning the window size

The window is the one parameter, and it trades context completeness against
noise. Too small and you have reintroduced the small-chunk problem; too large
and you are back to returning a page.

In [9]:
for w in [0, 1, 3, 5, 8]:
    r = sentence_window(QUESTION, window=w, top_n=1)[0]
    print(f"window=+/-{w}: {len(r['window']):>5} chars | {r['window'][:110]}...")
    print()

window=+/-0:    93 chars | The Cat seemed to think that there was enough of it now in sight, and no more of it appeared....

window=+/-1:  1026 chars | “It’s no use speaking to it,” she thought, “till its ears have come, or at least one of them.” In another minu...

window=+/-3:  2004 chars | Alice began to feel very uneasy: to be sure, she had not as yet had any dispute with the Queen, but she knew t...

window=+/-5:  3871 chars | The chief difficulty Alice found at first was in managing her flamingo: she succeeded in getting its body tuck...

window=+/-8:  5185 chars | “Oh, hush!” the Rabbit whispered in a frightened tone. You see, she came rather late, and the Queen said—”

“G...



Rules of thumb:

- `window=0` is plain sentence retrieval - sharp and usually unanswerable.
- `window=2-4` is the useful range for prose.
- `window>6` approaches paragraph retrieval and starts pulling in unrelated
  material from adjacent passages.

Structured text (API docs, legal clauses) tolerates smaller windows because each
unit is more self-contained. Narrative prose needs larger ones.

### 6. Answer quality across window sizes

Now the measurement that actually matters: does the LLM answer better?

In [10]:
def answer_with_window(question, window, top_n=3):
    results = sentence_window(question, window=window, top_n=top_n)
    context = "\n\n".join(f"[{i}] {r['window']}" for i, r in enumerate(results))
    answer = ask(
        "Answer the question using ONLY the context below. If the context is "
        "incomplete, say exactly what is missing.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return len(context), answer


for w in [0, 3, 6]:
    n_chars, answer = answer_with_window(QUESTION, window=w)
    print("=" * 72)
    print(f"window = +/-{w}   ({n_chars} characters of context)")
    print("=" * 72)
    print(answer)
    print()

window = +/-0   (961 characters of context)
Based on the context provided, the Cheshire Cat disappears by fading away. Specifically, the text states that "The Cat’s head began fading away the moment he was gone, and, by the time he had come back with the Duchess, it had entirely disappeared."



  [retry 1] RateLimitError - sleeping 5s


window = +/-3   (5129 characters of context)
Based on the context provided, the Cheshire Cat disappears by fading away. Specifically, the text states that "The Cat’s head began fading away the moment he was gone, and, by the time he had come back with the Duchess, it had entirely disappeared."



  [retry 1] RateLimitError - sleeping 5s


  [retry 2] RateLimitError - sleeping 10s


window = +/-6   (9854 characters of context)
Based on the context provided, the Cheshire Cat disappears by fading away. Specifically, the text states that "The Cat’s head began fading away the moment he was gone, and, by the time he had come back with the Duchess, it had entirely disappeared."



### 7. Sentence window versus parent document

| | sentence window | parent document |
|---|---|---|
| indexed unit | one sentence | small child chunk |
| returned unit | match +/- N sentences | the whole parent |
| context shape | centred on the match | fixed block |
| crosses document boundaries | yes, unless you guard it | no, by construction |
| best for | flowing prose, transcripts, books | documents with real structure (sections, tickets, pages) |
| storage | one ordered list | vector store + docstore |

**The boundary caveat is the important one.** Our `sentences` list was built from
one book, so splicing across a paragraph boundary is harmless. In a multi-document
corpus, a naive window will happily join the end of document A to the start of
document B and hand the LLM a Frankenstein passage. Always store a document id
per sentence and clamp the window to its own document.

In [11]:
# The production-shaped version: sentences carry their source, windows are clamped.
sent_records = []
for parent_id, parent in enumerate(parents):
    for local_pos, s in enumerate(SENT_RE.split(parent)):
        s = s.strip()
        if len(s) > 25:
            sent_records.append({"text": s, "doc_id": parent_id, "pos": len(sent_records)})

by_doc = {}
for rec in sent_records:
    by_doc.setdefault(rec["doc_id"], []).append(rec)

safe_vectors = build_index([r["text"] for r in sent_records])


def safe_window(query, window=3, top_n=3):
    """Window that never crosses a source-document boundary."""
    hits = search(safe_vectors, [r["text"] for r in sent_records], query, top_n=top_n)
    out = []
    for pos, score in hits:
        rec = sent_records[pos]
        siblings = by_doc[rec["doc_id"]]
        idx = siblings.index(rec)
        lo, hi = max(0, idx - window), min(len(siblings), idx + window + 1)
        out.append((rec["doc_id"], score,
                    " ".join(s["text"] for s in siblings[lo:hi])))
    return out


for doc_id, score, text in safe_window(QUESTION, window=3, top_n=2):
    print(f"doc_{doc_id} (cosine={score:.3f}) - window stays inside this document")
    print("  ", text[:200], "...")
    print()

doc_164 (cosine=0.615) - window stays inside this document
   The Cat’s head began fading away the moment he was gone, and, by the time he had come back with the Duchess, it had entirely disappeared; so the King and the executioner ran wildly up and down looking ...

doc_154 (cosine=0.596) - window stays inside this document
   She was looking about for some way of escape, and wondering whether she could get away without being seen, when she noticed a curious appearance in the air: it puzzled her very much at first, but, aft ...



### 8. Key takeaways

- Sentence windows give the **sharpest possible index** and centre the returned
  context on the actual match.
- Store sentences as one ordered list; the window is defined by position.
- Guard against **overlapping windows** (wasted slots) and **cross-document
  windows** (spliced nonsense).
- `window = 2-4` is the useful range for prose; tune it by reading answers, not
  by reading scores.

Next: [`04_contextual_chunk_headers.ipynb`](04_contextual_chunk_headers.ipynb) -
instead of changing what you *return*, change what you *embed*.